# Baseline — Concepts Guessing (20 Questions)

**Competition:** each game is a "20 questions" dialogue — a series of yes/no questions
about a hidden **concept** (from `concepts.txt`). Given the questions and answers of a
test game, submit your **10 best guesses** ranked.

- **Task:** rank concepts per game (`guess_1` … `guess_10`)
- **Metric:** rewards having the true concept early in your 10 guesses
- **Kaggle link:** _TODO: add link_

**Approach:** treat every game as a text document — concatenate its questions, each
tagged with its yes/no answer — then TF-IDF + Logistic Regression over the concept
classes. The predicted probabilities give a natural top-10 ranking.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape, train["concept"].nunique(), "concepts")

(89583, 5) (11864, 4) 124 concepts


In [2]:
# One document per game: "question <YES/NO>" tokens keep the answer's meaning
def to_docs(df):
    d = df.copy()
    d["qa"] = d["question"].str.lower() + " ans_" + d["answer"].str.lower()
    return d.groupby("game_id")["qa"].apply(" ".join)

train_docs = to_docs(train)
train_labels = train.groupby("game_id")["concept"].first().loc[train_docs.index]
test_docs = to_docs(test)
print(len(train_docs), "train games,", len(test_docs), "test games")

3720 train games, 496 test games


In [3]:
# Validation: hold out 20% of games
Xtr, Xva, ytr, yva = train_test_split(train_docs, train_labels, test_size=0.2,
                                      random_state=0, stratify=train_labels)
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=2000, C=5.0),
)
model.fit(Xtr, ytr)

proba = model.predict_proba(Xva)
classes = model.classes_
order = np.argsort(-proba, axis=1)[:, :10]
top1  = np.mean(classes[order[:, 0]] == yva.values)
top10 = np.mean([y in classes[o] for y, o in zip(yva.values, order)])
mrr   = np.mean([1.0 / (list(classes[o]).index(y) + 1) if y in classes[o] else 0.0
                 for y, o in zip(yva.values, order)])
print(f"top-1: {top1:.3f}   top-10: {top10:.3f}   MRR@10: {mrr:.3f}")

top-1: 0.105   top-10: 0.470   MRR@10: 0.203


In [4]:
# Retrain on all games, build the ranked submission
model.fit(train_docs, train_labels)
proba = model.predict_proba(test_docs)
classes = model.classes_
order = np.argsort(-proba, axis=1)[:, :10]

sub = pd.DataFrame({"game_id": test_docs.index})
for i in range(10):
    sub[f"guess_{i+1}"] = classes[order[:, i]]
sub.to_csv("submission.csv", index=False)
sub.head()

,game_id,guess_1,guess_2,guess_3,guess_4,guess_5,guess_6,guess_7,guess_8,guess_9,guess_10
0,test_00001,doctor,police officer,owl,pillow,pencil,dream,bus,spider,chef,smartphone
1,test_00002,television,egg,carrot,butterfly,lemonade,snow,bicycle,soccer ball,chef,desert
2,test_00003,t-shirt,backpack,shoes,water,guitar,apple,egg,chocolate,glasses,star
3,test_00004,mountain,sun,beach,cloud,library,island,pilot,farm,shark,firefighter
4,test_00005,eagle,butterfly,goldfish,police officer,pencil,ice cream,motorcycle,owl,rabbit,lion


## Ideas to improve

- **Sentence embeddings** (MiniLM) of questions instead of TF-IDF — questions with the
  same meaning but different words ("does it fly?" / "can it be airborne?") should match.
- Model each turn's contribution: a "yes" on a rare question is highly informative —
  weight features by answer and question rarity.
- An LLM can play the guesser directly: feed it the dialogue and the concept list and
  ask for a ranked shortlist.
